In [1]:
ALADIN_KEY = "ttbseoll770145001"
PINECONE_KEY = "pcsk_1JaXF_NB96Zvye9cmG7jnfHAsQy2odgSZ6fefruZ3bcQHMBNbWWQCk3zCbW6niNC2AEfH"

In [2]:
import requests
from typing import List, Dict, Optional

class AladinBookFetchService:
    BASE_URL = "https://www.aladin.co.kr/ttb/api/ItemList.aspx"
    LOOKUP_URL = "https://www.aladin.co.kr/ttb/api/ItemLookUp.aspx"
    
    def __init__(self, ttb_key: str):
        self.ttb_key = ttb_key

    """
    알라딘 API를 호출해 인기책 / 베스트셀러 리스트를 가져옴
    
    :param query_type: API 호출 타입 (Bestseller, ItemNewAll, ItemNewSpecial, BlogBest 등)
    :param max_results: 가져올 책 개수 (1~100)
    :param category_id: 검색할 카테고리 ID. 0이면 전체. (예: 판타지 1101)
    :param search_target: 검색 대상 (Book, Music, DVD 등)
    :param output: 응답 포맷 (js, xml 등)
    :param version: API 버전
    :return: 책 정보 리스트
    """
    def fetch_books(
        self,
        query_type: str = "Bestseller",
        max_results: int = 10,
        search_target: str = "Book",
        category_id: int = 0,
        output: str = "js",
        version: str = "20131101"
    ) -> Optional[List[Dict]]:
        params = {
            "ttbkey": self.ttb_key,
            "QueryType": query_type,
            "MaxResults": max_results,
            "SearchTarget": search_target,
            "CategoryId": category_id,
            "output": output,
            "Version": version,
        }

        try:
            response = requests.get(self.BASE_URL, params=params)
            response.raise_for_status()
            data = response.json()

            items = data.get("item", [])
            books = []
            
            for item in items:
                book = { # 기본 정보
                    "title": item.get("title"),
                    "author": item.get("author"),
                    "publisher": item.get("publisher"),
                    "cover": item.get("cover"),
                    "isbn13": item.get("isbn13"),
                }
                
                isbn13 = item.get("isbn13") # 상세 정보 추가
                if isbn13:
                    detail = self.fetch_book_detail(isbn13)
                    if detail:
                        book.update(detail)  # description, category 추가
                
                books.append(book)
                
            return books

        except requests.RequestException as e:
            print(f"❌ API 호출 오류: {e}")
            return None

    """
    알라딘 API를 통해 특정 도서의 상세 정보를 가져옴
    """
    def fetch_book_detail(self, isbn13: str):
        params = {
            "ttbkey": self.ttb_key,
            "itemIdType": "ISBN13",
            "ItemId": isbn13,
            "output": "js",
            "Version": "20131101",
            "Cover": "Big",
            "OptResult": "cateList,fulldescription,authors"
        }

        try:
            response = requests.get(self.LOOKUP_URL, params=params)
            response.raise_for_status()
            data = response.json()
    
            if "item" not in data or not data["item"]:
                return None
    
            item = data["item"][0]
            
            category_text = item.get("categoryName")
            category_parts = [c.strip() for c in category_text.split(">")][1:]

            return {
                "description": item.get("description", ""),
                "category": category_parts
            }
        
        except requests.RequestException as e:
            print(f"❌ 상세 조회 오류 ({isbn13}): {e}")
            return None

In [3]:
from transformers import AutoTokenizer, AutoModel
import torch

class E5Embedding:
    def __init__(self):
        """
        HuggingFace 모델이랑 토크나이저 같은거 다 불러오자
        참고로 tokenizer은.. 문자를 모델의 입력으로 바꿔주는 도구래! 
        미리 학습되어있는거 불러오면 좋겠지?
        """
        self.device = "cpu" # "cuda" if torch.cuda.is_available() else "cpu"
        # self.tokenizer = AutoTokenizer.from_pretrained("intfloat/e5-small") # model_name 넣어주면 알아서 불러와줌
        # self.model = AutoModel.from_pretrained("intfloat/e5-small")
        self.tokenizer = AutoTokenizer.from_pretrained("intfloat/multilingual-e5-small")
        self.model = AutoModel.from_pretrained("intfloat/multilingual-e5-small")

    # 입력 텍스트 생성 (타이틀 + 설명 + 저자 등 결합)
    def build_text(self, row):
        categories = ", ".join(row['category'])
        parts = [
            f"Title: {row['title']}\n",
            f"Category: {categories}\n",
            f"Description: {row['description']}"
        ]
        return " ".join( # 리스트의 문자열들을 공백으로 연결할건데.....
            [p for p in parts if isinstance(p, str)] # NaN이나 None이 있으면 제외함
        ) # 최종적으로 하나의 문장 형태로 반환한다고 함!! "Title: ... Author: ... Publisher: ... Description: ..."

        # title = row.get("title", "")
        # desc = row.get("description", "")
        
        # # category는 리스트니까 자연어로 변환
        # if isinstance(row.get("category"), list):
        #     category = ", ".join(row["category"])
        # else:
        #     category = row.get("category", "")
        
        # # 자연어 문서로 변환
        # return f"{title} 은(는) {category} 장르의 책으로, {desc}"

    def embed_batch(self, batch):
        """
        embed_batch 함수:
        책 텍스트 리스트를 받아 E5 임베딩을 batch 단위로 만들어 반환함
        얘는 대량으로 처리하기 좋은 방법이라서..... 뭐 정기적으로 대량임베딩하거나 할때 좋대
        나중에 e5를 다른걸로 바꾸거나 뭐.... 모델 운영하다보면 semantic search가 부정확해지거나 할때 쓰면 굿

        어차피 우린 제목, 저자, 출판사, 설명 정도만 있어서... SBERT나 OpenAI emb 같은거 굳이...
        """
        self.model.to(self.device).eval()
        batch_texts = [f"passage: {t}" for t in batch["text"]] # 각 텍스트 앞에 passage:를 붙힘!(e5 권장; 문맥 signal)

        inputs = self.tokenizer(
            batch_texts, return_tensors="pt", 
            truncation=True, # 길이 넘치면 자름
            padding=True,  # batch 단위로 다른 문장 길이에 맞게 padding
            max_length=256 # 최대 토큰 길이 제한
        ).to(self.device)

        with torch.no_grad():  # gradient 비활성화
            outputs = self.model(**inputs) # 딕셔너리를 언패킹(**)하여 모델에 전달
        
            # 원래는 아웃풋이 다 토큰단위인데.... mean해줘서 문장단위로 임베딩하게 된다는뎅
            emb = outputs.last_hidden_state.mean(dim=1)  # mean pooling
            emb = torch.nn.functional.normalize(emb, p=2, dim=1) # 정규화

        # pytorch 텐서는 기본적으로 연산 그래프를 추적해서 back-prop을 계산하나봐
        # 근데 .cpu().numpy()는 오직 gradient 추적 없는 순수 값(텐서)만 가능한 OP라서
        # .detach()를 통해서 그래프를 끊고 순수 값으로 탈바꿈 시킨대
        emb = emb.detach().cpu().numpy().tolist()
        return {"embedding": emb}

    def embed_query(self, query_text: str):
        # E5 모델의 쿼리 형식을 명시
        self.model.to(self.device).eval()
        formatted_query = f"query: {query_text}" 
        
        # embed_batch를 재활용하거나 새로 정의
        # 여기서는 간단히 새로운 함수를 만들어서 query: 접두사를 붙여줍니다.
        inputs = self.tokenizer(
            [formatted_query], return_tensors="pt",  
            truncation=True, padding=True, max_length=256
        ).to(self.device)
    
        # (이하 기존 embed_batch의 임베딩 로직 동일)
        with torch.no_grad():
            outputs = self.model(**inputs)
            emb = outputs.last_hidden_state.mean(dim=1)
            emb = torch.nn.functional.normalize(emb, p=2, dim=1)
    
        return emb.detach().cpu().numpy().tolist()[0]

In [4]:
from pinecone import Pinecone, ServerlessSpec

class VectorDBService:
    def __init__(self, api_key: str, 
                 index_name: str = "books-index", 
                 dimension: int = 384):
        """
        이 클래스로 말하자면,,, pinecone을 편하게 쓰기 위한 wrapper임!!
        Pinecone 초기화 및 인덱스 연결
        """
        self.pc = Pinecone(api_key=api_key)
        self.index_name = index_name

        # 인덱스 없으면 생성
        if index_name not in self.pc.list_indexes().names():
            self.pc.create_index(
                name=index_name,
                dimension=dimension,
                metric="cosine",
                spec=ServerlessSpec(cloud="aws", region="us-east-1")
            )

        # 인덱스 연결
        self.client = self.pc.Index(index_name)

    def reset_index(self):
        """ DB에 리셋 """
        self.client.delete(delete_all=True)
        print("Pinecone index has been reset!")
    
    def add_vectors(self, ids: list[str], embeddings: list[list[float]], metadata: list[dict]):
        """ 벡터와 메타데이터를 DB에 저장 """
        vectors = [
            {
                "id": i,
                "values": e,
                "metadata": m
            }
            for i, e, m in zip(ids, embeddings, metadata)
        ]
        
        self.client.upsert(vectors=vectors)

    def query(self, embedding: list[float], top_k: int = 5):
        """ 쿼리 벡터로 DB에서 유사 벡터 검색 """
        return self.client.query(vector=embedding, top_k=top_k, include_metadata=True)

In [5]:
aladin_service = AladinBookFetchService(ttb_key=ALADIN_KEY)
e5_service = E5Embedding()
pinecone_service = VectorDBService(api_key=PINECONE_KEY)

# 책 불러오기
bestsellers = aladin_service.fetch_books(query_type="Bestseller", max_results=100)

# 임베딩 하기
# 배치 텍스트와 book_id를 같은 순서로 저장하니 배치단위로 처리해도 매칭 안전함
# batch_texts = ["book A text", "book B text", "book C text"]
batch_texts = []
book_ids = []
for idx, book in enumerate(bestsellers, start=1):
    print(f"{idx}. {book['title']} - {book['author']} ({book['publisher']})")
    print(f"=> {book['category']}")
    print(f"=> {book['description'][:100]}... \n")
    text = e5_service.build_text(book)
    batch_texts.append(text)
    book_ids.append(book['isbn13']) # 고유 ID, DB에 key로 사용

embeddings = e5_service.embed_batch({"text": batch_texts})["embedding"]
for emb in embeddings:
    print(f"{len(emb)}: {emb[:5]}")

/home/0uk/.local/lib/python3.10/site-packages/huggingface_hub/file_download.py:942: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(


1. 어스탐 경의 임사전언 - 이영도 (지은이) (황금가지)
=> ['소설/시/희곡', '판타지/환상문학', '한국판타지/환상소설']
=> 한국 단행본 출판 수출 역사를 뒤바꾸며 전 세계 17개 언어권 30여 개 나라에서 인기리에 출간되고 있는 『눈물을 마시는 새』의 저자, 이영도 작가의 신작 장편소설 『어스탐 경의 ... 

2. 팬텀 버스터즈 5 - S코믹스 - 네오쇼코 (지은이), 김지혜 (옮긴이) (㈜소미미디어)
=> ['만화', '본격장르만화', '판타지', '액션 판타지']
=> 갑자기 나타난 수수께끼의 훈남 음양사, 카데노코지 보탄. 그의 목적은 시시쿠노 가문에 대한 선전 포고와 모가리를 복종시키는 것이었다! 보탄에 의해 대량의 악령을 받아들이게 된 모가... 

3. 체인소 맨 21 - 후지모토 타츠키 (지은이) (학산문화사(만화))
=> ['만화', '본격장르만화', '판타지', '액션 판타지']
=> ... 

4. 트렌드 코리아 2026 - 2026 대한민국 소비트렌드 전망 - 김난도, 전미영, 최지혜, 권정윤, 한다혜, 이혜원, 이수진, 서유현, 전다현, 이준영, 이향은, 김나은 (지은이) (미래의창)
=> ['경제경영', '트렌드/미래전망', '트렌드/미래전망 일반']
=> 세상은 작용과 반작용, 치열한 정반합(正反合)의 소용돌이가 거세게 휘몰아치고 있다. 방향을 잡기 어려울 정도로 속도가 빠르고 정신이 없다. 그렇다면 우리의 방향타는 어디에 있는가?... 

5. 내가 키운 S급들 1~2 + 굿즈 패키지 세트 - 비완 (지은이), 근서 (원작), seri (각색) (JAYPLEMEDIA)
=> ['만화', '인터넷 연재 만화']
=> &lt;사망한 한유현의 스킬과 능력치가 두 배로 전이됩니다.&gt; 그리고 힘과 함께 동생의 기억도 흘러들어 온다. 마지막 보답의 지속 시간이 끝나 간다. 시간이 되면 라우치타스의... 

6. 처음 만나는 양자의 세계 - 양자 역학부터 양자 컴퓨터 까지 - 채은미 (지은이) (북플레저)
=> ['과학',

In [6]:
pinecone_service.reset_index()
print(pinecone_service.client.describe_index_stats())

Pinecone index has been reset!
{'dimension': 384,
 'index_fullness': 0.0,
 'metric': 'cosine',
 'namespaces': {},
 'total_vector_count': 0,
 'vector_type': 'dense'}


In [7]:
ids = book_ids
embs = embeddings
meta = bestsellers

pinecone_service.add_vectors(ids, embs, meta)

In [8]:
# test_emb = embeddings[0]

# user_query = "드래곤이 나오고 마법사가 등장하는 판타지 소설을 추천해 줘."
# user_query = "정통 판타지 소설, 드래곤, 마법사, 이세계 모험 관련 책" -> 일단 이걸로 채택
# user_query = "Title: 드래곤과 마법사의 세계 Category: 소설/시/희곡, 판타지 Description: 영웅이 드래곤을 무찌르거나 마법 학교에서 수련하는 내용의 책"

# user_query = "그 비스크 돌은 사랑을 한다, 만화, 순정만화, 레이디스 코믹, 키타가와 마린과 고죠 와카나, 두 사람의 관계는 마침내 사귀게 된 마린과 와카나. 와카나 생일에 만나서는 어느 행동을 둘러싸고 말다툼을 벌이는데…?!"
user_query = "Title: 그 비스크 돌은 사랑을 한다 Category: 만화, 순정만화, 레이디스 코믹 Description: 키타가와 마린과 고죠 와카나, 두 사람의 관계는 마침내 사귀게 된 마린과 와카나. 와카나 생일에 만나서는 어느 행동을 둘러싸고 말다툼을 벌이는데…?!"

test_emb = e5_service.embed_query(user_query) # 수정된 함수 사용
results = pinecone_service.query(test_emb, top_k = 5)['matches']
for result in results:
    print(result)

{'id': '9791138488044',
 'metadata': {'author': '후쿠다 신이치 (지은이), 박연지 (옮긴이)',
              'category': ['만화', '순정만화', '레이디스 코믹'],
              'cover': 'https://image.aladin.co.kr/product/37625/78/coversum/k392032049_1.jpg',
              'description': '키타가와 마린과 고죠 와카나, 두 사람의 관계는 마침내 사귀게 된 마린과 와카나. '
                             '와카나 생일에 만나서는 어느 행동을 둘러싸고 말다툼을 벌이는데…?! 더구나 두 사람은 '
                             '추억이 깃든 그곳에 가서….',
              'isbn13': '9791138488044',
              'publisher': '㈜소미미디어',
              'title': '그 비스크 돌은 사랑을 한다 15 - S코믹스, 초판 종료'},
 'score': 0.950518727,
 'values': []}
{'id': '9791142332876',
 'metadata': {'author': '아즈마 키요히코 (지은이)',
              'category': ['만화', '코믹/명랑만화'],
              'cover': 'https://image.aladin.co.kr/product/37561/39/coversum/k362032425_1.jpg',
              'description': '조금씩 성장하는 아이와, 변함없는 어른들. 하루하루 즐거운 기억을 혹은 추억을 '
                             '만들어가는 요츠바의 일상. 계절은 어느덧 가을을 넘어 겨울을 향해 가고, 크리스마스를 '
                             '

In [9]:
def build_query_text(user_query):
    return (
        "This is a search query for books in the fantasy genre.\n"
        "Interpret the query as describing the desired book's content and theme.\n"
        f"User query: {user_query}"
    )
user_query = "드래곤 나오고 마법사 등장하는 판타지 소설 추천해줘"
query_text = build_query_text(user_query)
query_vector = e5_service.embed_batch({"text": [query_text]})["embedding"][0]

results = pinecone_service.query(query_vector, top_k = 5)['matches']
for result in results:
    print(result)

{'id': '9791139657494',
 'metadata': {'author': '비완 (지은이), 근서 (원작), seri (각색)',
              'category': ['만화', '인터넷 연재 만화'],
              'cover': 'https://image.aladin.co.kr/product/37726/40/coversum/k482033167_1.jpg',
              'description': '&lt;사망한 한유현의 스킬과 능력치가 두 배로 전이됩니다.&gt; 그리고 힘과 함께 '
                             '동생의 기억도 흘러들어 온다. 마지막 보답의 지속 시간이 끝나 간다. 시간이 되면 '
                             '라우치타스의 체액이 뿜어내는 독기로 나도 죽을 것이다. &lt;무슨 소원이든 한 가지 '
                             '들어주는 소원석이 주어집니다!&gt; “……줘, 시간을 되돌려 줘. 내 동생이 죽기 '
                             '전으로.”',
              'isbn13': '9791139657494',
              'publisher': 'JAYPLEMEDIA',
              'title': '내가 키운 S급들 1~2 + 굿즈 패키지 세트'},
 'score': 0.816377223,
 'values': []}
{'id': '9791193638859',
 'metadata': {'author': '김난도, 전미영, 최지혜, 권정윤, 한다혜, 이혜원, 이수진, 서유현, 전다현, 이준영, '
                        '이향은, 김나은 (지은이)',
              'category': ['경제경영', '트렌드/미래전망', '트렌드/미래전망 일반'],
              'cover': 'https://image.alad

In [10]:
for idx, book in enumerate(bestsellers):
    print(f"{idx}: {book['title']}, {book['category']}")

0: 어스탐 경의 임사전언, ['소설/시/희곡', '판타지/환상문학', '한국판타지/환상소설']
1: 팬텀 버스터즈 5 - S코믹스, ['만화', '본격장르만화', '판타지', '액션 판타지']
2: 체인소 맨 21, ['만화', '본격장르만화', '판타지', '액션 판타지']
3: 트렌드 코리아 2026 - 2026 대한민국 소비트렌드 전망, ['경제경영', '트렌드/미래전망', '트렌드/미래전망 일반']
4: 내가 키운 S급들 1~2 + 굿즈 패키지 세트, ['만화', '인터넷 연재 만화']
5: 처음 만나는 양자의 세계 - 양자 역학부터 양자 컴퓨터 까지, ['과학', '물리학', '양자역학']
6: 박곰희 연금 부자 수업 - 4개의 통장으로 월 300만 원 만들기, ['경제경영', '재테크/투자', '재테크/투자 일반']
7: 절창, ['소설/시/희곡', '한국소설', '2000년대 이후 한국소설']
8: 위버멘쉬 - 누구의 시선도 아닌, 내 의지대로 살겠다는 선언, ['인문학', '인문 에세이']
9: 이영도 필사노트 vol.1 - 후회는 부정된 자신에의 그리움, ['소설/시/희곡', '판타지/환상문학', '한국판타지/환상소설']
10: 대형주 추세추종 투자법칙, ['경제경영', '재테크/투자', '주식/펀드']
11: 크리스티안 페촐트, ['예술/대중문화', '영화/드라마', '영화감독/배우']
12: 혼모노, ['소설/시/희곡', '한국소설', '2000년대 이후 한국소설']
13: 사탄탱고 - 2025 노벨문학상 수상, ['소설/시/희곡', '세계의 소설', '동유럽소설']
14: 기분이 태도가 되지 않게 (헬로키티 에디션) - 기분 따라 행동하다 손해 보는 당신을 위한 심리 수업, ['에세이', '한국에세이']
15: 이상한 문장 그만 쓰는 법 - 어휘, 좋은 표현, 문장 부호까지 한 번에, ['인문학', '책읽기/글쓰기', '글쓰기']
16: 영어책 한 권 외워봤니? 뉴 에디션 - 딱 한 권만 넘으면 영어 울렁증이 사라진다, ['자기계발', 